# 🚀 Multi-Query RAG: Scalable Medical Vector Database Builder (50k–100k Docs)

This notebook builds a high-performance, persistent **ChromaDB vector database** for our **Multi-Query RAG System** using Google Colab's free T4 GPU.

### 🎯 Capabilities:
1. **Multi-Dataset Support**: Easily select between **SciFact** (5,183 docs), **NFCorpus** (3,633 medical docs), or **TREC-COVID** (scaled up to **50,000 to 100,000 documents**).
2. **GPU Batch Encoding**: Uses `BAAI/bge-base-en-v1.5` (768-dim) with PyTorch batch size 256 for fast encoding (~12-15 min for 100k docs).
3. **Pure ChromaDB Storage**: HNSW cosine index (`hnsw:space = cosine`) with chunked insertion (`batch_size = 2,000`) to prevent RAM memory spikes.
4. **Sanity Check**: Verifies single vs multi-query retrieval with Reciprocal Rank Fusion (RRF) and redundancy metrics.
5. **One-Click Export**: Automatically packages into `scifact_vector_db.zip` and triggers download directly to your machine.

> 💡 **Local Deployment**: Once downloaded, simply unzip `scifact_vector_db.zip` into the `scifact_vector_db/` folder in your project root!

In [ ]:
# Step 1: Install required dependencies
!pip install -q beir sentence-transformers chromadb tqdm

In [ ]:
# Step 2: Verify GPU acceleration
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ Running on CPU. For 50k-100k docs, go to Runtime -> Change runtime type -> T4 GPU for 15x faster vectorization!")

Using device: cuda
GPU Name: Tesla T4
VRAM Available: 15.64 GB


In [ ]:
# Step 3: Dataset Configuration (Scale to 50k–100k documents)
# Options:
# 1. 'scifact'    - 5,183 scientific papers (fast benchmark)
# 2. 'nfcorpus'   - 3,633 medical nutrition & clinical queries
# 3. 'trec-covid' - 171,332 clinical papers (set MAX_DOCS to 50000 or 100000)

DATASET_NAME = "trec-covid"  # Change to 'trec-covid' or 'nfcorpus' as needed
MAX_DOCS = None          # Set to 50000 or 100000 for large datasets, or None for full corpus

print(f"Selected Dataset: {DATASET_NAME}")
print(f"Max Documents Cap: {MAX_DOCS if MAX_DOCS else 'Full Corpus'}")

Selected Dataset: trec-covid
Max Documents Cap: Full Corpus


In [ ]:
# Step 4: Download & Load Corpus from BEIR
import os, json, pathlib
from beir import util
from beir.datasets.data_loader import GenericDataLoader

url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{DATASET_NAME}.zip"
out_dir = "./datasets"
data_path = util.download_and_unzip(url, out_dir)

# Load corpus, test queries, and ground-truth relevance labels
corpus, queries, qrels = GenericDataLoader(data_folder=data_path).load(split="test")

# Cap documents if MAX_DOCS is specified
if MAX_DOCS and len(corpus) > MAX_DOCS:
    print(f"Subsampling corpus from {len(corpus):,} down to {MAX_DOCS:,} documents...")
    # Ensure relevant documents for test queries are preserved in the sample
    relevant_doc_ids = set()
    for qid, docs in qrels.items():
        relevant_doc_ids.update(docs.keys())

    sample_keys = list(relevant_doc_ids)
    for k in corpus.keys():
        if len(sample_keys) >= MAX_DOCS:
            break
        if k not in relevant_doc_ids:
            sample_keys.append(k)
    corpus = {k: corpus[k] for k in sample_keys}

print(f"\n✅ Successfully loaded {DATASET_NAME.upper()}!")
print(f"Indexed documents: {len(corpus):,}")
print(f"Test queries:      {len(queries):,}")
print(f"Qrels entries:     {len(qrels):,}")

/usr/local/lib/python3.13/dist-packages/beir/util.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


  0%|          | 0/171332 [00:00<?, ?it/s]


✅ Successfully loaded TREC-COVID!
Indexed documents: 171,332
Test queries:      50
Qrels entries:     50


In [ ]:
# Step 5: Load BAAI/bge-base-en-v1.5 Embedding Model
from sentence_transformers import SentenceTransformer

model_name = "BAAI/bge-base-en-v1.5"
print(f"Loading {model_name} on {device}...")
embed_model = SentenceTransformer(model_name, device=device)
embedding_dim = embed_model.get_sentence_embedding_dimension()
print(f"✅ Embedding Model Ready! Output Dimension: {embedding_dim}")

Loading BAAI/bge-base-en-v1.5 on cuda...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Embedding Model Ready! Output Dimension: 768


/tmp/ipykernel_17525/3168022486.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_dim = embed_model.get_sentence_embedding_dimension()


In [ ]:
import numpy as np
from tqdm.auto import tqdm

doc_ids = list(corpus.keys())
# Semantic passage representation: Title + Abstract
doc_texts = [
    (corpus[doc_id].get("title", "").strip() + "\n" + corpus[doc_id].get("text", "").strip()).strip()
    for doc_id in doc_ids
]

# Batch size 256 delivers maximum throughput on Colab T4 GPU (16GB VRAM)
batch_size = 256 if device == "cuda" else 32  # Reduced batch_size to 256 to avoid OutOfMemoryError
print(f"Encoding {len(doc_texts):,} passages with batch_size={batch_size}...")

doc_embeddings = embed_model.encode(
    doc_texts,
    batch_size=batch_size,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
)

print(f"✅ Generated embeddings shape: {doc_embeddings.shape} ({doc_embeddings.nbytes / 1e6:.1f} MB)")

Encoding 171,332 passages with batch_size=256...


Batches:   0%|          | 0/670 [00:00<?, ?it/s]

✅ Generated embeddings shape: (171332, 768) (526.3 MB)


In [ ]:
# Step 7: Build Persistent ChromaDB Store (Chunked Upserts)
import chromadb
import shutil

chroma_dir = "./scifact_chroma_db"
if os.path.exists(chroma_dir):
    shutil.rmtree(chroma_dir)

client = chromadb.PersistentClient(path=chroma_dir)
collection = client.create_collection(
    name="scifact_collection",
    metadata={"hnsw:space": "cosine"}
)

# Ingest in chunks of 2,000 to maintain low RAM usage during index construction
chunk_size = 2000
print(f"Ingesting {len(doc_ids):,} documents into ChromaDB in chunks of {chunk_size}...")
for i in tqdm(range(0, len(doc_ids), chunk_size), desc="Populating ChromaDB"):
    batch_ids = doc_ids[i:i + chunk_size]
    batch_texts = doc_texts[i:i + chunk_size]
    batch_embeddings = doc_embeddings[i:i + chunk_size].tolist()
    batch_metadatas = [
        {"title": corpus[did].get("title", "")[:250], "doc_id": str(did)}
        for did in batch_ids
    ]

    collection.add(
        ids=[str(did) for did in batch_ids],
        documents=batch_texts,
        embeddings=batch_embeddings,
        metadatas=batch_metadatas
    )

print(f"✅ ChromaDB created successfully with {collection.count():,} indexed records!")

Ingesting 171,332 documents into ChromaDB in chunks of 2000...


Populating ChromaDB:   0%|          | 0/86 [00:00<?, ?it/s]

✅ ChromaDB created successfully with 171,332 indexed records!


In [ ]:
# Step 8: Export Evaluation Queries and Ground-Truth Labels
eval_dir = "./scifact_eval"
os.makedirs(eval_dir, exist_ok=True)

with open(os.path.join(eval_dir, "queries.json"), "w", encoding="utf-8") as f:
    json.dump(queries, f, indent=2)

with open(os.path.join(eval_dir, "qrels.json"), "w", encoding="utf-8") as f:
    json.dump(qrels, f, indent=2)

print(f"✅ Exported {len(queries):,} evaluation queries and {len(qrels):,} qrel sets!")

✅ Exported 50 evaluation queries and 50 qrel sets!


In [ ]:
# Step 9: Multi-Query Retrieval Sanity Check
query_instruction = "Represent this sentence for searching relevant passages: "

sample_query = "0-dimensional biomaterials show inductive properties."
multi_queries = [
    sample_query,
    "zero-dimensional nanomaterials inductive properties",
    "electromagnetic induction in low dimensional biomaterials",
    "quantum dot biomaterial induction mechanisms"
]

print("--- Testing Single Query Retrieval ---")
sq_emb = embed_model.encode([query_instruction + sample_query], normalize_embeddings=True).tolist()
sq_res = collection.query(query_embeddings=sq_emb, n_results=5)
print(f"Single query returned {len(sq_res['ids'][0])} docs: {sq_res['ids'][0]}")

print("\n--- Testing Multi-Query Retrieval ---")
mq_embs = embed_model.encode([query_instruction + q for q in multi_queries], normalize_embeddings=True).tolist()
mq_res = collection.query(query_embeddings=mq_embs, n_results=5)

all_retrieved = []
for i, q in enumerate(multi_queries):
    docs = mq_res["ids"][i]
    all_retrieved.extend(docs)
    print(f"  Q{i+1}: '{q[:40]}...' -> {docs}")

unique_docs = list(dict.fromkeys(all_retrieved))
redundant_count = len(all_retrieved) - len(unique_docs)
redundancy_pct = (redundant_count / len(all_retrieved)) * 100

print(f"\n📊 Multi-Query Analysis:")
print(f"  Total Candidates Retrieved: {len(all_retrieved)}")
print(f"  Unique Passages Retained:   {len(unique_docs)}")
print(f"  Redundant Filtered:         {redundant_count} ({redundancy_pct:.1f}% redundancy rate)")
print("✅ Sanity verification passed!")

--- Testing Single Query Retrieval ---
Single query returned 5 docs: ['gv96hlw6', 'etwri0on', '6jgao58w', 'vuyfl8jc', 'o6vido4j']

--- Testing Multi-Query Retrieval ---
  Q1: '0-dimensional biomaterials show inductiv...' -> ['gv96hlw6', 'etwri0on', '6jgao58w', 'vuyfl8jc', 'o6vido4j']
  Q2: 'zero-dimensional nanomaterials inductive...' -> ['k8hjsbmi', 'seedc4zh', 'xp4mcxui', '9hfl770o', 'jvwzcpgt']
  Q3: 'electromagnetic induction in low dimensi...' -> ['cirr6zmg', 'q241z9u0', 'vuyfl8jc', 'khinou6x', 'ral378t8']
  Q4: 'quantum dot biomaterial induction mechan...' -> ['brz1fn2h', 'vuyfl8jc', 'an1oegqj', 'khinou6x', '7jz3qpcw']

📊 Multi-Query Analysis:
  Total Candidates Retrieved: 20
  Unique Passages Retained:   17
  Redundant Filtered:         3 (15.0% redundancy rate)
✅ Sanity verification passed!


In [ ]:
# Step 10: Compress Vector Database & Auto-Download
import zipfile
from google.colab import files

zip_filename = "scifact_vector_db.zip"
print(f"Compressing database directories into {zip_filename}...")

with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, filenames in os.walk("./scifact_chroma_db"):
        for filename in filenames:
            filepath = os.path.join(root, filename)
            arcname = os.path.relpath(filepath, ".")
            zipf.write(filepath, arcname)

    for root, dirs, filenames in os.walk("./scifact_eval"):
        for filename in filenames:
            filepath = os.path.join(root, filename)
            arcname = os.path.relpath(filepath, ".")
            zipf.write(filepath, arcname)

zip_size_mb = os.path.getsize(zip_filename) / (1024 * 1024)
print(f"✅ Successfully created {zip_filename} ({zip_size_mb:.2f} MB)!")
print("Downloading to your machine...")
files.download(zip_filename)

### 🎉 Local Machine Setup:
1. Move `scifact_vector_db.zip` into your project directory.
2. Unzip so that `scifact_vector_db/scifact_chroma_db` is present.
3. Launch your app with `streamlit run app.py` or run `python test_end_to_end.py`!